In [ ]:
!pip install torchtuples pycox
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.3/141.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.7/413.7 kB 22.8 MB/s eta 0:00:00
  Created wheel for feather-format: filename=feather_format-0.4.1-py3-none-any.whl size=2435 sha256=5a6981ca8b5dce077382cac33a03d2c6ee9dba44704b458b78f1abd3caf6d018
  Stored in directory: /root/.cache/pip/wheels/77/5b/0e/0e63d10b6353208a085a321ea2eed2578

### Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np

def encode_columns(column):
    """
    Encodes categorical columns using a predefined mapping strategy.
    Converts 'Positive', 'Yes' to 1, 'Negative', 'No' to 0, and handles missing values.
    """
    column_filled = column.fillna('NA').astype(str)  # Handle missing values
    custom_mapping = {
        'Positive': 1, 'Negative': 0, 'Yes': 1, 'No': 0,
        'Not done': 2, 'NA': -1  # Special handling for missing values
    }
    return column_filled.map(custom_mapping)

def preprocess_features(df, reference_columns=None):
    """
    Preprocesses a DataFrame by handling categorical features, encoding values,
    and filling missing values. Ensures consistency with reference columns.

    Args:
        df (pd.DataFrame): The DataFrame to preprocess.
        reference_columns (list, optional): The list of final columns from training data.

    Returns:
        pd.DataFrame: The processed DataFrame with aligned columns.
    """
    # Apply mappings for specific categorical features
    mappings = {
        'dri_score': {'Low': 1, 'Intermediate': 2, 'Intermediate - TED AML case <missing cytogenetics': 2.5,
                      'High': 3, 'High - TED AML case <missing cytogenetics': 3.5, 'Very high': 4,
                      'N/A - disease not classifiable': -1, 'N/A - non-malignant indication': -2,
                      'N/A - pediatric': -2, 'TBD cytogenetics': -1, 'Missing disease status': -1},
        'psych_disturb': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'cyto_score': {'Favorable': 1, 'Intermediate': 2, 'Normal': 3, 'Other': 4,
                       'Poor': 5, 'TBD': 6, 'Not tested': 6, 'Unknown': 7},
        'diabetes': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'arrhythmia': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'vent_hist': {'No': 0, 'Yes': 1, 'Unknown': 2},
        'renal_issue': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'pulm_severe': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'hepatic_mild': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'cmv_status': {"-/+": 3, "+/-": 2, "-/-": 1, "+/+": 0, "Missing": -1},
        'conditioning_intensity': {'MAC': 5, 'RIC': 4, 'NMA': 3, 'TBD': 2,
                                   'No drugs reported': 1, 'N/A, F(pre-TED) not submitted': 0, 'Missing': -1},
        'cyto_score_detail': {'Poor': 4, 'TBD': 3, 'Not tested': 2, 'Intermediate': 1, 'Favorable': 0, 'Missing': -1},
        'graft_type': {'Bone marrow': 0, 'Peripheral blood': 1},
        'tce_match': {'Permissive': 0, 'Fully matched': 1, 'GvH non-permissive': 2, 'HvG non-permissive': 3, 'NA': -1},
        'sex_match': {'F-F': 1, 'M-M': 2, 'M-F': 3, 'F-M': 4, 'NA': -1}
    }

    # Apply mappings after filling missing values
    for col, mapping in mappings.items():
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').map(mapping)

    # Fill missing numerical values using median
    numerical_cols = [
        'hla_match_c_high', 'hla_high_res_8', 'hla_low_res_6',
        'hla_high_res_6', 'hla_high_res_10', 'hla_match_dqb1_high',
        'hla_match_c_low', 'hla_match_drb1_low', 'hla_match_dqb1_low'
    ]
    for col in numerical_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # Handle missing values exactly as in your notebook
    df['rituximab'] = df['rituximab'].fillna(df['rituximab'].mode()[0]).map({'Yes': 1, 'No': 0})
    df['mrd_hct'] = encode_columns(df['mrd_hct'])
    df['in_vivo_tcd'] = encode_columns(df['in_vivo_tcd'])
    df['hepatic_severe'] = encode_columns(df['hepatic_severe'])
    df['prior_tumor'] = encode_columns(df['prior_tumor'])
    df['peptic_ulcer'] = encode_columns(df['peptic_ulcer'])
    df['rheum_issue'] = encode_columns(df['rheum_issue'])

    # Categorical columns that were explicitly one-hot encoded in your notebook
    categorical_cols = [
        'tbi_status', 'prim_disease_hct', 'obesity', 'prod_type',
        'ethnicity', 'race_group', 'gvhd_proph', 'tce_imm_match',
        'tce_div_match', 'donor_related', 'melphalan_dose', 'cardiac',
        'pulm_moderate'
    ]

    # Fill missing values before encoding
    for col in ['ethnicity', 'obesity']:
        if col in df.columns:
            df[col] = df[col].fillna("Missing")

    for col in ['tce_imm_match', 'tce_div_match', 'cardiac', 'pulm_moderate']:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")

    # Apply one-hot encoding
    df = pd.get_dummies(df, columns=[col for col in categorical_cols if col in df.columns], dtype=int)

    # Ensure test dataset aligns with training columns
    if reference_columns is not None:
        for col in reference_columns:
            if col not in df.columns:
                df[col] = 0  # Add missing columns
        df = df[reference_columns]  # Ensure column order matches

    # Convert True/False values to 1/0
    df = df.applymap(lambda x: 1 if x is True else (0 if x is False else x))

    return df


train_file_path = "1oCgZEJBCPQAJc9Jc2UVJtLBejSmMlR9F"
test_file_path = "1L-XeJlshuo0UkZ902VqQWw1T224BUoxZ"
train_url = f"https://drive.google.com/uc?export=download&id={train_file_path}"
test_url = f"https://drive.google.com/uc?export=download&id={test_file_path}"
train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

train_df = preprocess_features(train_df)
final_train_columns = train_df.columns.tolist()
test_df = preprocess_features(test_df, final_train_columns)

<ipython-input-1-2a37e7eb7da1>:104: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: 1 if x is True else (0 if x is False else x))
<ipython-input-1-2a37e7eb7da1>:104: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: 1 if x is True else (0 if x is False else x))


### XGBoost Model

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index


# Drop non-feature columns
X = train_df.drop(columns=['efs', 'efs_time', 'ID'])

# Clean column
def sanitize_feature_names(df):
    df.columns = [str(col).replace('[', '').replace(']', '').replace('<', '').replace(' ', '_') for col in df.columns]
    return df

X = sanitize_feature_names(X)

# Extract target variables
y_time = train_df['efs_time']
y_event = train_df['efs']

# ----------------------
# Kaplan-Meier Transformation to Calculate Survival Probabilities
# ----------------------
def transform_survival_probability(df, time_col='efs_time', event_col='efs'):
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    y = kmf.survival_function_at_times(df[time_col]).values
    return y

# Calculate Kaplan-Meier survival probabilities
train_df["y"] = transform_survival_probability(train_df, time_col='efs_time', event_col='efs')

# Drop the columns used for target variable calculation (efs_time, efs)
train_cleaned = train_df.drop(columns=['efs_time', 'efs'])

# ----------------------
# Preprocessing: Remove Low-Variance Features
# ----------------------
selector = VarianceThreshold(threshold=0.05)
X_reduced = selector.fit_transform(X)

# Get selected features
selected_features = selector.get_support(indices=True)
X = X.iloc[:, selected_features]

# Train-Test Split
X_train, X_val, y_time_train, y_time_val, y_event_train, y_event_val = train_test_split(
    X, y_time, y_event, test_size=0.2, random_state=42
)

# ----------------------
# XGBoost AFT Model
# ----------------------
# Prepare AFT bounds for XGBoost
lower_bound = y_time_train.values.copy()
upper_bound = np.where(y_event_train == 1, y_time_train, np.inf)

dtrain = xgb.DMatrix(X_train, label_lower_bound=lower_bound, label_upper_bound=upper_bound)
dval = xgb.DMatrix(X_val)

params = {
    'objective': 'survival:aft',
    'eval_metric': 'aft-nloglik',
    'aft_loss_distribution': 'normal',
    'aft_loss_distribution_scale': 1.0,
    'tree_method': 'hist',
    'learning_rate': 0.01,
    'max_depth': 8,
}

model_xgb = xgb.train(params, dtrain, num_boost_round=300)
pred_time_xgb = model_xgb.predict(dval)
risk_xgb = 1 / pred_time_xgb

# C-index for XGBoost
c_xgb = concordance_index(y_time_val, -risk_xgb, y_event_val)
print(f"XGBoost C-index: {c_xgb:.3f}")


XGBoost C-index: 0.671
